In [1]:
from bs4 import BeautifulSoup as bs
from Bio import Entrez
import xmltodict
import urllib.request
from urllib.request import urlopen
import requests
import re
import pandas as pd
import csv
import openpyxl
import time
import numpy as np
from xlsxwriter.utility import xl_rowcol_to_cell
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

import warnings
warnings.filterwarnings('ignore')

/tmp/ipykernel_9333/1986711241.py:8: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# DATA FROM CANCER.GOV. Get drug names and corresponding page link

In [2]:
def get_drug_list(link,fname):
    r=requests.get(link).text
    s=bs(r,'lxml')
    article=s.find('article')
    if article != None:
    #print(article)
        d={'Drug_name':'Link'}
        a= article.find_all('ul', class_='no-bullets no-description')
        for i in a:
            for q in i.find_all('a',href=True):
                if len(q.get_text()) >1:
                    d[q.get_text()]=q['href'].rsplit('/', 1)[-1]
                    
    
    first_iter=True
    for i in d:
        if first_iter:
            first_iter = False
            continue
        q=d[i]
        d[i]= 'cancer.gov/about-cancer/treatment/drugs/' + q
        if 'selumetinib sulfate' in d[i]:
            d[i]= 'cancer.gov/about-cancer/treatment/drugs/selumetinibsulfate'
    for i in d:
        print(i+':'+d[i])
            
    df=pd.DataFrame.from_dict(d,orient='index')
    df.to_excel(fname,header=False)

In [4]:
#Parse Cancer.gov and get drug list
get_drug_list('https://www.cancer.gov/about-cancer/treatment/drugs','cancer_gov_drug_list.xlsx')
print('cancer_gov_drug_list Saved')

Drug_name:Link
Abecma (Idecabtagene Vicleucel):cancer.gov/about-cancer/treatment/drugs/idecabtagenevicleucel
Abemaciclib:cancer.gov/about-cancer/treatment/drugs/abemaciclib
Abiraterone Acetate:cancer.gov/about-cancer/treatment/drugs/abirateroneacetate
Abraxane (Paclitaxel Albumin-stabilized Nanoparticle Formulation):cancer.gov/about-cancer/treatment/drugs/nanoparticlepaclitaxel
ABVD:cancer.gov/about-cancer/treatment/drugs/abvd
ABVE:cancer.gov/about-cancer/treatment/drugs/abve
ABVE-PC:cancer.gov/about-cancer/treatment/drugs/abve-pc
AC:cancer.gov/about-cancer/treatment/drugs/ac
Acalabrutinib:cancer.gov/about-cancer/treatment/drugs/acalabrutinib
AC-T:cancer.gov/about-cancer/treatment/drugs/ac-t
Actemra (Tocilizumab):cancer.gov/about-cancer/treatment/drugs/tocilizumab
Adagrasib:cancer.gov/about-cancer/treatment/drugs/adagrasib
Adcetris (Brentuximab Vedotin):cancer.gov/about-cancer/treatment/drugs/brentuximabvedotin
ADE:cancer.gov/about-cancer/treatment/drugs/ade
Ado-Trastuzumab Emtansine:c

# For each link from above, extract Drug_name, FDA_approved, Tumor_types, Details of Condition for which FDA approved, NCI Drug dictionary definition link.

In [6]:
DEBUG = False
def extract_drugname_cancerlist_nci_link(link):
    #print(links)
    url='https://www.'+link.replace(' ', '-')
    #url=links
    print(url)
    page = urlopen(url)
    #print(page)
    soup = bs(page)

    article = soup.find('article')
    #print(article)
    #Drug name in h1 tag
    drug=article.find('h1').text

    #FDA approved or not 
    fda_app = ""
    fda_sec = article.find_all('div', class_='two-columns brand-fda')
    if fda_sec :
        if DEBUG : print('FDA section found')
        for sec in fda_sec:
            if DEBUG : 
                print('\t',sec.text)
                print('\t',sec.find('div', class_='column1'))
            if 'FDA' in sec.find('div', class_='column1').text:
                if DEBUG : print('\tFOUND FDA')
                fda_app = sec.find('div',class_='column2').text
    
    #Get drug usage details now
    #print('##################################')
    paras = article.find("div",class_='accordion')
    #Variable is set to true if details found in the page
    found = False
    drug_details = ""
    nci_ddd_link=""
    ct_link=""
    cancer_list=[]
    for p in paras.findChildren(recursive=False):
        if DEBUG :
            print('----')
            print(p)
        #Need to capture data between heading "Use in Cancer" and "More About"
        #Discard capturing the other data
        if found and "more about" in p.text.lower():
            #print(':::::::::::',p)
            found = False
            if DEBUG : print('---End of drug details')
        #NCI link
        if not found and "nci drug dictionary" in p.text.lower():
            #print('$$$$$$$$$$$$$$$ Link ::: ',p)
            nci_ddd_link= p.find('a')['href']
            #print(nci_site)
            #break
        elif not found and "find clinical trials" in p.text.lower():
            #print('--------Clinical trials-----')
            #print(p.find('a')['href'])
            ct_link = p.find('a')['href']
            break
        if found :
            if DEBUG : 
                print(p.name)
                print(p.text)
            if p.name != 'a' and p.name !='strong':
                if p.findChildren() and p.name != 'p':
                    if DEBUG : print('******In find children******')
                    for x in p.findChildren():
                        if DEBUG : 
                            print('\t',x.name)
                            print('\t',x.text)
                        if x.name != 'a' and x.name !='strong':
                            #Remove extra spaces in the text and then append
                            txt = ' '.join(x.text.replace('\n', ' ').replace('\r', ' ').split())
                            drug_details += "\n" + txt
                            #print('\t',x.text)
                            #print('\t',x.name)
                        elif x.name == 'strong':
                            #All disease mentions are bold with strong tag
                            cancer_list.append(x.text)
                else:
                    if DEBUG : print('******In else of find children******')
                    
                    #Remove extra spaces in the text and then append
                    txt = ' '.join(p.text.replace('\n', ' ').replace('\r', ' ').split())
                    drug_details += "\n" + txt
                    #print(p.text)
                    #print(p.name)
            #This else not necessary. need to check
            elif p.name == 'strong':
                #All disease mentions are bold with strong tag
                cancer_list.append(p.text)
        elif "use in cancer" in p.text.lower():
            #set the flag to true as page has drrug usage details
            found = True
            if DEBUG : print('---Found drug details')
    if DEBUG :
        print('-------------------------------------------')
        print(fda_app,'\n',drug,'\n',cancer_list,'\n',drug_details)
        print('-------------------------------------------')
    return drug.strip(), fda_app.strip(), cancer_list, drug_details.strip(), nci_ddd_link.strip(),ct_link.strip()



In [7]:
links_list = pd.read_excel('cancer_gov_drug_list.xlsx', usecols=['Link'])
drug_info_df = pd.DataFrame(columns=['Drug_name', 'Link', 'FDA_approved', 'Tumor_types', 'Details_of_Condition_for_which_FDA_approved', 'NCI_Drug_Dictionary_Definition_link','Clinical_trials_link'])

for link in links_list['Link']:
    drug, fda_app, cancer_list, drug_details, nci_ddd_link, ct_link = extract_drugname_cancerlist_nci_link(link)
    drug_details = drug_details.replace("'", "").replace(",", " ")
    #fda_app = str(fda_app)[2:-2]
    clist = str(cancer_list)[2:-2].replace("'", "").replace(".", "")
    drug_info_df.loc[len(drug_info_df)] = {'Drug_name': drug, 'Link': link, 'FDA_approved': fda_app, 'Tumor_types': clist, 
                                           'Details_of_Condition_for_which_FDA_approved': drug_details, 
                                           'NCI_Drug_Dictionary_Definition_link': nci_ddd_link, 'Clinical_trials_link':ct_link}

#display(drug_info_df)
drug_info_df.to_excel('NCI_drug_tumor_list.xlsx', index=False)
print('NCI_drug_tumor_list Saved')

https://www.cancer.gov/about-cancer/treatment/drugs/idecabtagenevicleucel
https://www.cancer.gov/about-cancer/treatment/drugs/abemaciclib
https://www.cancer.gov/about-cancer/treatment/drugs/abirateroneacetate
https://www.cancer.gov/about-cancer/treatment/drugs/nanoparticlepaclitaxel
https://www.cancer.gov/about-cancer/treatment/drugs/abvd
https://www.cancer.gov/about-cancer/treatment/drugs/abve
https://www.cancer.gov/about-cancer/treatment/drugs/abve-pc
https://www.cancer.gov/about-cancer/treatment/drugs/ac
https://www.cancer.gov/about-cancer/treatment/drugs/acalabrutinib
https://www.cancer.gov/about-cancer/treatment/drugs/ac-t
https://www.cancer.gov/about-cancer/treatment/drugs/tocilizumab
https://www.cancer.gov/about-cancer/treatment/drugs/adagrasib
https://www.cancer.gov/about-cancer/treatment/drugs/brentuximabvedotin
https://www.cancer.gov/about-cancer/treatment/drugs/ade
https://www.cancer.gov/about-cancer/treatment/drugs/ado-trastuzumab-emtansine
https://www.cancer.gov/about-canc

In [8]:
def getNCI_Thesaurus_Link(driver,url):
    print(url)
    driver.get(url)
    time.sleep(1)
    soup = bs(driver.page_source)
    #print(bs)
    nci_link = soup.find('a',class_="dictionary-definiton__nci-thesaurus-link")
    if nci_link:
        print(nci_link['href'])
        #print(df)
        return nci_link['href']
    else:
        return ""
        
def populateNCI_Thesaurus_Link():
    options = Options()
    options.add_argument('--disable-application-cache')
    options.add_argument('--headless')
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    
    service = Service()
    driver = webdriver.Chrome(service=service, options=options)

    df = pd.read_excel('NCI_drug_tumor_list.xlsx')
    df = df.fillna("")
    for i, row in df.iterrows():
        print('--------------------------')
        nci_def_link = row['NCI_Drug_Dictionary_Definition_link']
        #print(nci_def_link)
        nci_link = getNCI_Thesaurus_Link(driver,nci_def_link)
        df.at[i,'NCI_Thesaurus_Link'] = nci_link
    display(df)
    df.to_excel('NCI_drug_tumor_list_Thesaurus_link.xlsx', index=False)
    print('NCI_drug_tumor_list_Thesaurus_link Saved after populating Thesaurus link')
populateNCI_Thesaurus_Link()

--------------------------
https://www.cancer.gov/publications/dictionaries/cancer-drug/def/763837
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C117729
--------------------------
https://www.cancer.gov/publications/dictionaries/cancer-drug/def/706364
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C97660
--------------------------
https://www.cancer.gov/publications/dictionaries/cancer-drug/def/552704
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C68845
--------------------------
https://www.cancer.gov/publications/dictionaries/cancer-drug/def/38690
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C2688
--------------------------
https://www.cancer.gov/publications/dictionaries/cancer-drug/def/39364
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C9509
--------------------------
https://www

,Drug_name,Link,FDA_approved,Tumor_types,Details_of_Condition_for_which_FDA_approved,NCI_Drug_Dictionary_Definition_link,Clinical_trials_link,NCI_Thesaurus_Link
0,Idecabtagene Vicleucel,cancer.gov/about-cancer/treatment/drugs/idecab...,Yes,Multiple myeloma,Idecabtagene vicleucel is approved to treat:\n...,https://www.cancer.gov/publications/dictionari...,https://www.cancer.gov/research/participate/cl...,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...
1,Abemaciclib,cancer.gov/about-cancer/treatment/drugs/abemac...,Yes,Breast cancer,Abemaciclib is approved to treat:\nBreast canc...,https://www.cancer.gov/publications/dictionari...,https://www.cancer.gov/research/participate/cl...,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...
2,Abiraterone Acetate,cancer.gov/about-cancer/treatment/drugs/abirat...,Yes,Prostate cancer,Abiraterone acetate is approved to be used wit...,https://www.cancer.gov/publications/dictionari...,https://www.cancer.gov/research/participate/cl...,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...
3,Paclitaxel Albumin-stabilized Nanoparticle For...,cancer.gov/about-cancer/treatment/drugs/nanopa...,Yes,"Breast cancer, Non-small cell lung cancer, Pan...",Paclitaxel albumin-stabilized nanoparticle for...,https://www.cancer.gov/publications/dictionari...,https://www.cancer.gov/research/participate/cl...,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...
4,ABVD,cancer.gov/about-cancer/treatment/drugs/abvd,,Hodgkin lymphoma,ABVD is used to treat:\nHodgkin lymphoma.\nThi...,https://www.cancer.gov/publications/dictionari...,,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...
...,...,...,...,...,...,...,...,...
691,Idelalisib,cancer.gov/about-cancer/treatment/drugs/idelal...,Yes,Chronic lymphocytic leukemia,Idelalisib is approved to treat:\nChronic lymp...,https://www.cancer.gov/publications/dictionari...,https://www.cancer.gov/research/participate/cl...,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...
692,Ceritinib,cancer.gov/about-cancer/treatment/drugs/ceritinib,Yes,Non-small cell lung cancer,Ceritinib is approved to treat:\nNon-small cel...,https://www.cancer.gov/publications/dictionari...,https://www.cancer.gov/research/participate/cl...,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...
693,Loncastuximab Tesirine-lpyl,cancer.gov/about-cancer/treatment/drugs/loncas...,Yes,B-cell non-Hodgkin lymphoma,Loncastuximab tesirine-lpyl is approved to tre...,https://www.cancer.gov/publications/dictionari...,https://www.cancer.gov/research/participate/cl...,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...
694,Retifanlimab-dlwr,cancer.gov/about-cancer/treatment/drugs/retifa...,Yes,Merkel cell carcinoma,Retifanlimab-dlwr is approved to treat adults ...,https://www.cancer.gov/publications/dictionari...,https://www.cancer.gov/research/participate/cl...,https://ncit.nci.nih.gov/ncitbrowser/ConceptRe...


NCI_drug_tumor_list_Thesaurus_link Saved after populating Thesaurus link


# Extract Definition,Synonyms, Extranal source code from NCI Thesaurus

In [23]:
def populate_NCI_Thesaurus_Details():
    df = pd.read_excel('NCI_drug_tumor_list_Thesaurus_link.xlsx')
    df = df.fillna("")
    for i, row in df.iterrows():
        print('--------------------------')
        url = row['NCI_Thesaurus_Link']
        print(url)
        if url != '':
            page = requests.get(url)
            soup = bs(page.content, "html.parser")
            
            item_list = soup.find_all('p')
            for item in item_list: 
                text = item.text
                #print(text)
                if 'NCI Thesaurus Code' in text:
                    df.at[i,'NCI_Thesaurus_Code'] = text.split(":")[1].strip()
                elif 'Preferred Name' in text: 
                    #print('Preferred Name' in text)
                    df.at[i,'NCI_Preferred_Name'] =text.split(":")[1].strip()
                elif 'Display Name' in text: 
                    #print('Display Name' in text)
                    df.at[i,'NCI_Display_Name'] = text.split(":")[1].strip()
                elif 'Synonyms & Abbreviations' in text:
                    slist = item.find_all('td')
                    synonyms = ""
                    for syn in slist:
                        if synonyms.strip() != '':
                            synonyms = synonyms + "||"
                        synonyms = synonyms + syn.text
                    #print('----Synonyms ::',synonyms)
                    df.at[i,'NCI_Synonyms_and_Abbreviations'] = synonyms
                elif 'Definition' in text:
                    df.at[i,'NCI_Definition'] = text.split(":")[1].strip()
                elif 'External Source Codes' in text:
                    esc = ""
                    eslist = item.find_all('tr')
                    #print('$$$$$$$$$$$$$$$$$$')
                    for es in eslist:
                        es_tds = es.find_all('td')
                        #Remove extract breaks and spaces
                        tmp = es_tds[0].text.strip() +":"+es_tds[1].text.strip()
                        txt = ' '.join(tmp.replace('\n', ' ').replace('\r', ' ').split())
                        esc = esc.strip() + "\n"+ txt
                        if 'FDA UNII Code' in es_tds[0].text :
                            df.at[i,'FDA_UNII_Code'] = es_tds[1].text.strip()
                    df.at[i,'NCI_External_Source_Codes'] = esc
                    #print(esc)
    # Write the final DataFrame to Excel
    df.to_excel('NCI_drug_tumor_list_NCI_Thesaurus_Data.xlsx', index=False)
    print('NCI_drug_tumor_list_NCI_Thesaurus_Data Saved after populating NCI Thesaurus details')

populate_NCI_Thesaurus_Details()

--------------------------
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C117729
--------------------------
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C97660
--------------------------
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C68845
--------------------------
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C2688
--------------------------
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C9509
--------------------------
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C67162
--------------------------
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C67175
--------------------------
https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI%20Thesaurus&code=C63415
--------------------------
https:

# DATA FROM PUBCHEM

# The drug names in this file NCI_drug_tumor_list.xlsx is used to get SID from PubChem using esearch. It is saved in excel file SID_Drug_info.xlsx


In [13]:
def getsidandsource(drug):
    drug=drug.replace(" ","+")
    #print(drug)
    time.sleep(2)
    link = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstance&term="+drug+"&retmode=xml"
    print(link)
    page = urllib.request.urlopen(link)
    page= page.read().decode('utf-8')
    #print(page)
    esearch_result = xmltodict.parse(page)
    #print(esearch_result)
    sid=[]
    search = esearch_result['eSearchResult']['IdList']['Id']
    search=str(search)[1:-1]
    sid.append(search)
    #print(sid)
    return sid

In [14]:
drug_name = pd.read_excel('NCI_drug_tumor_list_NCI_Thesaurus_Data.xlsx', usecols=['Drug_name'])
drug_name = drug_name['Drug_name'].unique()
drug_sid_pc_df = pd.DataFrame(columns=['Drug_name', 'SID'])

for drug in drug_name:
    try:
        sid = getsidandsource(drug)
        sid = ','.join(sid)
        drug_sid_pc_df = pd.concat([drug_sid_pc_df, pd.DataFrame({'Drug_name': [drug], 'SID': [sid]})], ignore_index=True)
    except Exception as e:
        print('Exception:', drug, e)

drug_sid_pc_df.to_excel('SID_Drug_info.xlsx', index=False)
display(drug_sid_pc_df)
print('Saved SID_Drug_Info')

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstance&term=Idecabtagene+Vicleucel&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstance&term=Abemaciclib&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstance&term=Abiraterone+Acetate&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstance&term=Paclitaxel+Albumin-stabilized+Nanoparticle+Formulation&retmode=xml
Exception: Paclitaxel Albumin-stabilized Nanoparticle Formulation 'NoneType' object is not subscriptable
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstance&term=ABVD&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstance&term=ABVE&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstance&term=ABVE-PC&retmode=xml
Exception: ABVE-PC 'NoneType' object is not subscriptable
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pcsubstanc

,Drug_name,SID
0,Idecabtagene Vicleucel,"'482042220', '481101667', '472421408', '472420..."
1,Abemaciclib,"'489440693', '489440495', '489355322', '488985..."
2,Abiraterone Acetate,"'488957728', '488318359', '488305682', '488224..."
3,ABVD,"'135264548', '135264424', '127830723', '537882..."
4,ABVE,3533476
...,...,...
340,Vorinostat,"'488976341', '488427907', '488427906', '488427..."
341,XELIRI,"'483927056', '483926653', '319346856', '251914..."
342,XELOX,"'319408555', '312524939', '175266338', '135264..."
343,Ziv-Aflibercept,"'481985741', '481101909', '480401320', '381127..."


Saved SID_Drug_Info


# The SID in SID_Drug_info.xlsx is used to form a link using esummary to get PubChem_Synonym and source info column and saved in excel file Pubchem_synon.xlsx. Exception SID are saved in Exception_SID.xlsx 

In [31]:
def getsynonympubchem(sid):
    sid=sid.replace(" ","").replace("'","")
    #print(sid)
    #print(sid)
    time.sleep(1)
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id="+sid+"&retmode=xml"
    print(url)
    result = urllib.request.urlopen(url)
    data= result.read().decode('utf-8')
    entries = xmltodict.parse(data)
    print(entries)
    synonym=[]
    source_info=[]
    Id = entries ['eSummaryResult']['DocSum']['Id']
    #print(Id)
    for entry in entries ['eSummaryResult']['DocSum']['Item']:
        #print(entry)
        define = entry['@Name']
        #print(define)
        if define == 'SynonymList':
            extract=entry['Item']
            print(extract)
            for ext in extract:
                print(ext)
                text = {key: ext[key] for key in ext.keys() & {'#text'}}
                #print(text)
                text = list(text.values())
                text =str(text)[1:-1]
                text=text.replace("'","")
                #print(text)
                synonym.append(text)
        
        elif define == 'SourceNameList':
            extract2 = entry['Item']
            #print(extract2)
            for ext2 in extract2:
                #print(ext2)
                text2 = {key: ext2[key] for key in ext2.keys() & {'#text'}}
                #print(text)
                text2 = list(text2.values())
                text2 =str(text2)[1:-1]
                text2=text2.replace("'","")
                #print(text2)
                source_info.append(text2)
    print(source_info)
    print(synonym)
    print(len(synonym))
    return source_info, synonym
getsynonympubchem("485262066")
getsynonympubchem("485231341")


https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=485262066&retmode=xml
{'eSummaryResult': {'DocSum': {'Id': '485262066', 'Item': [{'@Name': 'SID', '@Type': 'Integer', '#text': '485262066'}, {'@Name': 'SourceNameList', '@Type': 'List', 'Item': [{'@Name': 'string', '@Type': 'String', '#text': '15699'}, {'@Name': 'string', '@Type': 'String', '#text': 'LGC Standards Product List'}, {'@Name': 'string', '@Type': 'String', '#text': 'LGC Standards'}]}, {'@Name': 'SourceID', '@Type': 'String', '#text': 'TRC-D288080'}, {'@Name': 'SourceCategoryList', '@Type': 'List', 'Item': {'@Name': 'string', '@Type': 'String', '#text': 'Chemical Vendors'}}, {'@Name': 'SourceReleaseDate', '@Type': 'Date', '#text': '2023/12/01 00:00'}, {'@Name': 'DepositDate', '@Type': 'Date', '#text': '2023/12/01 00:00'}, {'@Name': 'ModifyDate', '@Type': 'Date', '#text': '2023/12/01 00:00'}, {'@Name': 'RegistryNumber', '@Type': 'String', '#text': '122914-94-7'}, {'@Name': 'DBUrl', '@Type': 'String'

AttributeError: 'str' object has no attribute 'keys'

In [16]:
def getsynpc():
    df=pd.read_excel("SID_Drug_info.xlsx")
    df = df.set_index(['Drug_name']).apply(lambda x: x.str.split(',').explode()).reset_index()
    data_df=pd.DataFrame(columns = ['SID', 'Source_Info', 'PubChem_Synonym'])
    df.to_excel("split_sid_info.xlsx", index=False)
    exception_df=pd.DataFrame(columns=['SID'])
    for sid in df['SID']:
        try:
            source_info, synonym = getsynonympubchem(sid)
            source_info=str(source_info)[2:-2]
            source_info=source_info.replace("'","")
            synonym=str(synonym)[2:-2]
            synonym=synonym.replace("'","")
            data_df = pd.concat([data_df, pd.DataFrame({'SID': [sid], 'Source_Info': [source_info], 'PubChem_Synonym': [synonym]})], ignore_index=True)
        except Exception as e:
            print('Exception_SID:', sid, e)
            exception_df = pd.concat([exception_df, pd.DataFrame({'SID': [sid]})], ignore_index=True)
            
    data_df.to_excel("Final_pubchem.xlsx", index=False)
    display(data_df)
    exception_df.to_excel("Exception_SID.xlsx", index=False)
    print("Sync SID and Synonyms and saved final_pubchem.xlsx")
getsynpc()

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=482042220&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=481101667&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=472421408&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=472420314&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=472419851&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=463820146&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=404719648&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=384585532&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=489440693&retmode=xml
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pcsubstance&id=489440495&retmode=xml


,SID,Source_Info,PubChem_Synonym
0,'482042220',"26136, PubChem Reference Collection","24U06E3FHJ, ANTI-BCMA02 CAR LENTIVIRAL VECTOR,..."
1,'481101667',"26136, PubChem Reference Collection","Ide-cel, idecabtagen vicleucel, ABECMA, Idecab..."
2,'472421408',"2322, FDA Substance Registration System, FDA/S...","H4ON8SV8LK, ANTI-BCMA CAR T RECEPTOR, BB-2121 ..."
3,'472420314',"2322, FDA Substance Registration System, FDA/S...","24U06E3FHJ, ANTI-BCMA02 CAR LENTIVIRAL VECTOR,..."
4,'472419851',"2322, FDA Substance Registration System, FDA/S...","Ide-cel, ABECMA, Idecabtagene vicleucel, Anti-..."
...,...,...,...
4530,'485653895',"MMDB, MMDB, NCBI Protein Structures, NCBI Stru...","Zoledronic acid, ZOL"
4531,'485325631',"15699, LGC Standards Product List, LGC Standards","Zoledronic acid, 118072-93-8"
4532,'485325630',"15699, LGC Standards Product List, LGC Standards","Zoledronic-d3 Acid, 1134798-26-7"
4533,'485325629',"15699, LGC Standards Product List, LGC Standards","Zoledronic Acid-15N2,13C2, 1189694-79-8"


# Merge the drugname in the file split_sid_info.xlsx and PubChem synonyms in the file Final_pubchem.xlsx to get the column Drug_name and PubChem_Synonym in the file Pubchem_Data.xlsx

In [8]:
def merge():
    sid_list=pd.read_excel("split_sid_info.xlsx")
    display(sid_list)
    pubchem_synon=pd.read_excel("Final_pubchem.xlsx")
    display(pubchem_synon)
    drug_pubchem_synon=pd.merge(sid_list, pubchem_synon, how='inner', on='SID')
    drug_pubchem_synon=drug_pubchem_synon.drop(labels=['SID', 'Source_Info'], axis=1 )
    drug_pubchem_synon = drug_pubchem_synon.groupby('Drug_name')['PubChem_Synonym'].agg(lambda x: ', '.join(set(x))).reset_index()
    drug_pubchem_synon['PubChem_Synonym']=drug_pubchem_synon['PubChem_Synonym'].str.split(', ').apply(set).str.join(', ')
    drug_pubchem_synon.to_excel("Pubchem_Data.xlsx", index=False)
    display(drug_pubchem_synon)
merge()

,Drug_name,SID
0,Idecabtagene Vicleucel,'482042220'
1,Idecabtagene Vicleucel,'481101667'
2,Idecabtagene Vicleucel,'472421408'
3,Idecabtagene Vicleucel,'472420314'
4,Idecabtagene Vicleucel,'472419851'
...,...,...
5329,Zoledronic Acid,'485244526'
5330,Zoledronic Acid,'482316832'
5331,Zoledronic Acid,'482316831'
5332,Zoledronic Acid,'482316830'


,SID,Source_Info,PubChem_Synonym
0,'482042220',"26136, PubChem Reference Collection","24U06E3FHJ, ANTI-BCMA02 CAR LENTIVIRAL VECTOR,..."
1,'481101667',"26136, PubChem Reference Collection","Ide-cel, idecabtagen vicleucel, ABECMA, Idecab..."
2,'472421408',"2322, FDA Substance Registration System, FDA/S...","H4ON8SV8LK, ANTI-BCMA CAR T RECEPTOR, BB-2121 ..."
3,'472420314',"2322, FDA Substance Registration System, FDA/S...","24U06E3FHJ, ANTI-BCMA02 CAR LENTIVIRAL VECTOR,..."
4,'472419851',"2322, FDA Substance Registration System, FDA/S...","Ide-cel, ABECMA, Idecabtagene vicleucel, Anti-..."
...,...,...,...
4552,'485653895',"MMDB, MMDB, NCBI Protein Structures, NCBI Stru...","Zoledronic acid, ZOL"
4553,'485325631',"15699, LGC Standards Product List, LGC Standards","Zoledronic acid, 118072-93-8"
4554,'485325630',"15699, LGC Standards Product List, LGC Standards","Zoledronic-d3 Acid, 1134798-26-7"
4555,'485325629',"15699, LGC Standards Product List, LGC Standards","Zoledronic Acid-15N2,13C2, 1189694-79-8"


,Drug_name,PubChem_Synonym
0,ABVD,"COPP-ABVD, ABVD protocol, C522158, C034632, AB..."
1,AC,"N0001-008068, N0001-002416, N0001-004091, N000..."
2,ADE,"BOTOKI V MASK, HSDB 7774, Mioggi Healing Effec..."
3,Abemaciclib,"LY-2835219, Abemaciclib, BLP-003180, B0084-462..."
4,Abiraterone Acetate,"Abiraterone acetate (Standard), ABIRATERONE AC..."
...,...,...
321,XELIRI,"Xeliri, IRINOTECAN HYDROCHLORIDE [ORANGE BOOK]..."
322,XELOX,"CAPECITABINE [USP IMPURITY], Capecitabine Meda..."
323,Zanubrutinib,"1691249-45-2, 1633350-06-7, (R)-Zanubrutinib, ..."
324,Ziv-Aflibercept,"Aflibercept (USAN/INN), Vascular Endothelial G..."


# MERGE DATA FROM CANCER.GOV and PUBCHEM

In [25]:
def pubchemndncidatamerge():
    drug_info_df=pd.read_excel('NCI_drug_tumor_list_NCI_Thesaurus_Data.xlsx')
    pubchem_info=pd.read_excel('Pubchem_Data.xlsx')
    drug_pubchem_info=pd.merge(drug_info_df, pubchem_info, how='left', on='Drug_name')
    #display(drug_pubchem_info)
    #drug_pubchem_info=pd.merge(drug_pubchem_info, nci_thesaurus_info, how='left', left_on='Drug_name',)
    drug_pubchem_info= drug_pubchem_info.drop_duplicates()
    drug_pubchem_info.to_excel("Final_NCI_PubChem_Drug_Database.xlsx",index=False)
pubchemndncidatamerge()